# reshape, transpose et permute : suivre les mêmes cases

Comparer des transformations de même forme dont les coordonnées diffèrent.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#organiser-les-axes).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous utilisons une expérience indépendante pour voir les déplacements : les nombres 1 à 6 identifient les cases. Ils ne représentent pas des mesures audio.

| Opération | Nouvelles lignes | Que devient la valeur 4 ? |
| --- | --- | --- |
| Source (2,3) | [1,2,3] puis [4,5,6] | Ligne 1, colonne 0 |
| reshape(3,2) | [1,2] ; [3,4] ; [5,6] | Ligne 1, colonne 1 |
| transpose(0,1) | [1,4] ; [2,5] ; [3,6] | Ligne 0, colonne 1 |

reshape consomme les valeurs dans leur ordre logique ; transpose fait de chaque ancienne colonne une nouvelle ligne.

## Démonstration 1 · Vérifier les valeurs, puis une permutation de trois axes

Choisir l’opération d’après le sens, pas seulement la taille finale.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
# Expérience indépendante : identifiants de cases, pas mesures audio.


### Calculer l’état suivant

Le tableau a deux lignes et trois colonnes. Chaque valeur est distincte pour que son trajet reste visible.

In [4]:
X = torch.tensor([[1, 2, 3], [4, 5, 6]])


In [5]:
observer('X', X)

X forme (2, 3) type torch.int64 requires_grad False
  valeurs : [[1, 2, 3], [4, 5, 6]]


### Calculer l’état suivant

reshape(3,2) et transpose(0,1) donnent tous deux trois lignes et deux colonnes. Leur contenu diffère.

In [6]:
redecoupe = X.reshape(3, 2)
transpose = X.transpose(0, 1)


In [7]:
observer('redecoupe', redecoupe)
observer('transpose', transpose)

redecoupe forme (3, 2) type torch.int64 requires_grad False
  valeurs : [[1, 2], [3, 4], [5, 6]]
transpose forme (3, 2) type torch.int64 requires_grad False
  valeurs : [[1, 4], [2, 5], [3, 6]]


### Lire les sorties

La valeur de [1,2,3] devient celle de [1,3,2]. Elle reste 23, avec le même extrait, temps et fréquence.

In [8]:
print("original", X.tolist(), "forme", tuple(X.shape))
print("reshape", redecoupe.tolist(), "forme", tuple(redecoupe.shape))
print("transpose", transpose.tolist(), "forme", tuple(transpose.shape))


original [[1, 2, 3], [4, 5, 6]] forme (2, 3)
reshape [[1, 2], [3, 4], [5, 6]] forme (3, 2)
transpose [[1, 4], [2, 5], [3, 6]] forme (3, 2)


### Calculer l’état suivant

torch.arange(24) fabrique les identifiants 0 à 23. reshape(2,3,4) les organise en deux extraits, trois temps et quatre fréquences.

In [9]:
mel = torch.arange(24).reshape(2, 3, 4)


In [10]:
observer('mel', mel)

mel forme (2, 3, 4) type torch.int64 requires_grad False


### Calculer l’état suivant

permute(0,2,1) garde l’ancien axe 0 puis place l’ancien axe 2 avant l’ancien axe 1.

In [11]:
mel_permute = mel.permute(0, 2, 1)


In [12]:
observer('mel_permute', mel_permute)

mel_permute forme (2, 4, 3) type torch.int64 requires_grad False


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("axes exemple", tuple(mel.shape), "vers", tuple(mel_permute.shape))
print("case conservee", mel[1, 2, 3].item(), mel_permute[1, 3, 2].item())

axes exemple (2, 3, 4) vers (2, 4, 3)
case conservee 23 23


Une forme identique ne signifie pas une même association entre valeurs et coordonnées. Nous vérifions une case connue pour révéler le mauvais choix.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Une grille 2×3 est reshaped en 3×2. Quelle valeur trouve-t-on à la coordonnée [1,0] ?

In [14]:
exo_x = torch.arange(6).reshape(2,3)
exo_r = exo_x.reshape(3,2)
exo_valeur = ...  # Remplacer les points par votre réponse.
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_valeur == exo_r[1,0].item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

reshape conserve ici le parcours des cases 0,1,2,3,4,5.

</details>

### Exercice 2

Même grille initiale, mais avec transpose(0,1). Que vaut maintenant [1,0] ?

In [15]:
exo_x = torch.arange(6).reshape(2,3)
exo_t = exo_x.transpose(0,1)
exo_valeur = ...  # Remplacer les points par votre réponse.
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_valeur == exo_t[1,0].item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La nouvelle case [1,0] correspond à l’ancienne case [0,1].

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.arange(6).reshape(2,3)
exo_r = exo_x.reshape(3,2)
exo_valeur = 2
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_valeur == exo_r[1,0].item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.arange(6).reshape(2,3)
exo_t = exo_x.transpose(0,1)
exo_valeur = 1
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_valeur == exo_t[1,0].item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [16]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()